In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Gradient Descent and Curvature

A learning rate is usually found by trial: we try a value, watch the loss,
and lower the value if the loss blows up. Each training step moves against
the gradient of the loss, so the gradient fixes the direction of the step and
the learning rate sets only its distance. Yet a poor choice of distance makes
training either slow or unstable. How far can a step along the gradient be
trusted, and how many steps does that limit force on us?

A function of two variables already shows the difficulty. Take the bowl
$f(x, y) = \tfrac12(x^2 + 10\,y^2)$, which curves ten times faster along $y$
than along $x$, and start at the point $(2, 1)$, where $f = 7$ and the
gradient is $\nabla f = (2, 10)$. A step of size $\eta$ against the gradient
moves to $(2 - 2\eta,\, 1 - 10\eta)$. With $\eta = 0.01$ the value drops only
to $6.01$. With $\eta = 0.1$ it drops to $1.62$. With $\eta = 0.25$ it
*rises* to $12.4$. The direction was downhill every time; only the distance
differed.

How far a step can go depends on curvature. A step short enough for the
steep direction $y$ makes slow progress along the flat direction $x$. Even
the best fixed step needs $69$ iterations to shrink the error by a factor of
$10^6$, that is, to gain six digits. Stretch the ratio to $1000$, the
chapter's running example $\tfrac12(x_1^2 + 1000\,x_2^2)$, and it needs
$6{,}908$. Momentum brings that down to $315$. We work by hand on the opening
bowl, whose Hessian $\mathrm{diag}(1, 10)$ keeps every step checkable, and
quote counts for the stiffer running example.

We first bound the safe step through smoothness, which turns the local
linear model of that section into a guarantee.
We then solve gradient descent exactly on quadratics, where the ratio of
curvatures appears as the condition number, and show how momentum changes
the count. Three results carry the argument: the descent lemma
the equation, the optimal step
the equation, and the $\sqrt\kappa$ law of momentum
the equation. Throughout, the methods of this section only
respect the curvature. What a step can do once it models the curvature, as
Newton's method does, is the subject of
that section. Gradients are exact here, and the
noisy gradients of training wait for
that section. The code is plain NumPy, so
each update can be read against its equation.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Descent Directions and Smoothness

A gradient step answers two questions, which direction and how far. The first
has a clean answer from the first-order Taylor expansion. The second needs an
assumption about how fast the gradient changes. That assumption turns the
first-order approximation into a guarantee for a step of finite length.

### The Steepest Direction

The opening stepped along $-\nabla f$ without asking why. To see that it is
the right choice, expand $f$ to first order. For a differentiable
$f : \mathbb{R}^d \to \mathbb{R}$, a small step of size $\eta > 0$ along a
direction $\mathbf{d}$ changes the value by

$$
f(\mathbf{x} + \eta\,\mathbf{d}) = f(\mathbf{x}) + \eta\, \nabla f(\mathbf{x})^\top \mathbf{d} + o(\eta).
$$

To first order the change is $\eta$ times the directional derivative
$\nabla f(\mathbf{x})^\top \mathbf{d}$. Call $\mathbf{d}$ a **descent
direction** at $\mathbf{x}$ if $\nabla f(\mathbf{x})^\top \mathbf{d} < 0$.

**Proposition (descent directions, and the steepest one).** *Let $f$ be
differentiable at $\mathbf{x}$ with $\nabla f(\mathbf{x}) \neq \mathbf{0}$.*

1. *If $\nabla f(\mathbf{x})^\top \mathbf{d} < 0$, then
   $f(\mathbf{x} + \eta\,\mathbf{d}) < f(\mathbf{x})$ for every sufficiently
   small $\eta > 0$.*
2. *Among unit vectors, the directional derivative is minimized uniquely by the
   normalized negative gradient:*

$$
\min_{\|\mathbf{d}\| = 1} \nabla f(\mathbf{x})^\top \mathbf{d} = -\|\nabla f(\mathbf{x})\|,
\qquad \textrm{attained only at } \mathbf{d} = -\frac{\nabla f(\mathbf{x})}{\|\nabla f(\mathbf{x})\|}.
$$

**Proof.** For the first claim, the function
$\varphi(\eta) = f(\mathbf{x} + \eta\,\mathbf{d})$ has
$\varphi'(0) = \nabla f(\mathbf{x})^\top \mathbf{d} < 0$ by the chain rule, so
the difference quotient $(\varphi(\eta) - \varphi(0))/\eta$ is negative for all
small $\eta > 0$, that is, $\varphi(\eta) < \varphi(0)$. For the second, the
Cauchy--Schwarz inequality gives, for every unit vector $\mathbf{d}$,
$\nabla f(\mathbf{x})^\top \mathbf{d} \ge -\|\nabla f(\mathbf{x})\|\,\|\mathbf{d}\| = -\|\nabla f(\mathbf{x})\|$,
with equality only when $\mathbf{d}$ is a negative multiple of
$\nabla f(\mathbf{x})$. The only such unit vector is
$-\nabla f(\mathbf{x})/\|\nabla f(\mathbf{x})\|$, and it attains the bound.
$\blacksquare$

On the opening example the steepest unit direction at $(2, 1)$ is
$-(2, 10)/\sqrt{104} = (-0.196, -0.981)$, with slope
$-\|\nabla f\| = -10.198$. The axis direction $(-1, 0)$ is also downhill,
but its slope is only $-2$. Descent directions fill the open half-space of
vectors that make an acute angle with $-\nabla f$; the gradient picks the
steepest of them. All of this concerns slopes, the limit of short steps. For
a step of finite length, $\eta = 0.01$, we set the slope beside the actual
change along four unit directions.

In [ ]:
def f(v):
    return 0.5 * (v[0]**2 + 10 * v[1]**2)

def grad(v):
    return np.array([v[0], 10 * v[1]])

x = np.array([2.0, 1.0])
g = grad(x)
steepest = -g / np.linalg.norm(g)
candidates = [('steepest -grad/|grad|', steepest),
              ('diagonal -(1,1)/sqrt(2)', -np.ones(2) / np.sqrt(2)),
              ('axis     (-1,0)', np.array([-1.0, 0.0])),
              ('uphill   +(1,1)/sqrt(2)', np.ones(2) / np.sqrt(2))]
eta = 0.01
for name, d in candidates:
    print(f'{name}:  slope = {g @ d:+8.4f},  '
          f'f(x + eta d) - f(x) = {f(x + eta * d) - f(x):+.6f}')

At this step length the actual changes are $\eta$ times the slopes to within
one percent, so here the slopes are a fair guide. Differentiability promises
no more than that: a decrease for small enough steps. How small is the
question of the next subsection.

### Smoothness and the Descent Lemma

**Gradient descent** repeats the steepest step with a fixed **step size**
(learning rate) $\eta > 0$:

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \eta\, \nabla f(\mathbf{x}_k).
$$

Throughout this section $k$ counts iterations, and we abbreviate
$\mathbf{g}_k = \nabla f(\mathbf{x}_k)$. The first-order model justified the
direction. It says nothing about how far to trust it, because the gradient
at $\mathbf{x}_k$ stops describing $f$ once the step moves far enough for
the gradient to change. The assumption we need bounds that change.

**Definition ($L$-smoothness).** A differentiable $f$ is **$L$-smooth** if
its gradient changes by at most $L$ per unit of distance:

$$
\|\nabla f(\mathbf{x}) - \nabla f(\mathbf{y})\| \;\le\; L\, \|\mathbf{x} - \mathbf{y}\|
\qquad \textrm{for all } \mathbf{x}, \mathbf{y}.
$$

For twice continuously differentiable $f$ this is the same as every Hessian
eigenvalue lying in $[-L, L]$. Bounded eigenvalues give
the equation by integrating the Hessian along the segment
from $\mathbf{y}$ to $\mathbf{x}$, and the converse is a difference-quotient
limit (exercise 2a).
So $L$ bounds the curvature in every direction. The opening example has the
constant Hessian $\mathrm{diag}(1, 10)$, so $L = 10$. A function whose
curvature grows without limit has no such constant: $\tfrac14 x^4$ has
$f''(x) = 3x^2$, so no global $L$ exists. The backtracking line search
below handles that case.

Smoothness turns the first-order model into a one-sided guarantee, the
quadratic ceiling

$$
f(\mathbf{y}) \;\le\; f(\mathbf{x}) + \nabla f(\mathbf{x})^\top (\mathbf{y} - \mathbf{x}) + \tfrac{L}{2}\, \|\mathbf{y} - \mathbf{x}\|^2.
$$

The graph of $f$ never rises above its tangent plane by more than $L/2$
times the squared distance. Before proving it, insert a gradient step
$\mathbf{y} = \mathbf{x} - \eta\,\mathbf{g}$ and compare the two terms. The
first-order decrease $\eta\,\|\mathbf{g}\|^2$ grows linearly in $\eta$, while
the curvature error $\tfrac{L}{2}\eta^2\|\mathbf{g}\|^2$ grows quadratically.
For short steps the decrease dominates, and at $\eta = 2/L$ the two terms are
equal.

**Proposition (descent lemma).** *If $f$ is $L$-smooth, then the gradient step
the equation satisfies*

$$
f(\mathbf{x}_{k+1}) \;\le\; f(\mathbf{x}_k) - \eta \left(1 - \tfrac{L\eta}{2}\right) \|\mathbf{g}_k\|^2.
$$

*In particular $f$ strictly decreases whenever $0 < \eta < 2/L$ and
$\mathbf{g}_k \neq \mathbf{0}$, and the guaranteed decrease is largest at
$\eta = 1/L$, where it reads
$f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \tfrac{1}{2L}\|\mathbf{g}_k\|^2$.*

**Proof.** First the ceiling the equation. Restrict
$f$ to the segment, $\psi(t) = f(\mathbf{x} + t(\mathbf{y} - \mathbf{x}))$ for
$t \in [0, 1]$, so that
$\psi'(t) = \nabla f(\mathbf{x} + t(\mathbf{y}-\mathbf{x}))^\top (\mathbf{y} - \mathbf{x})$.
By the fundamental theorem of calculus and then Cauchy--Schwarz with the
Lipschitz bound the equation,

$$
\begin{aligned}
f(\mathbf{y}) - f(\mathbf{x}) - \nabla f(\mathbf{x})^\top (\mathbf{y}-\mathbf{x})
&= \int_0^1 \big(\nabla f(\mathbf{x} + t(\mathbf{y}-\mathbf{x})) - \nabla f(\mathbf{x})\big)^\top (\mathbf{y}-\mathbf{x})\, dt \\
&\le \int_0^1 L\,t\, \|\mathbf{y}-\mathbf{x}\|^2\, dt = \tfrac{L}{2}\|\mathbf{y}-\mathbf{x}\|^2.
\end{aligned}
$$

Now insert the gradient step $\mathbf{y} = \mathbf{x}_k - \eta\,\mathbf{g}_k$:

$$
f(\mathbf{x}_{k+1}) \;\le\; f(\mathbf{x}_k) - \eta\,\|\mathbf{g}_k\|^2 + \tfrac{L}{2}\,\eta^2\, \|\mathbf{g}_k\|^2,
$$

and the bracket $\eta(1 - L\eta/2)$ is positive exactly for $\eta < 2/L$ and
maximized at $\eta = 1/L$. $\blacksquare$

The proof compares gradients only between $\mathbf{x}_k$ and points of the
segment to $\mathbf{x}_{k+1}$, so it is enough that $\nabla f$ is
$L$-Lipschitz on that segment. On the opening example $\|\mathbf{g}\|^2 = 104$
and $L = 10$. At $\eta = 1/L = 0.1$ the lemma promises
$f \le 7 - 104/20 = 1.8$. At $\eta = 0.01$ it promises
$f \le 7 - 104 \cdot 0.01 \cdot 0.95 = 6.012$. At $\eta = 0.25 > 2/L$ the
bracket is negative and the bound exceeds the starting value, so the lemma
promises nothing. How much do these promises give away against the values
the three steps actually reach?

In [ ]:
x, L = np.array([2.0, 1.0]), 10.0
g = grad(x)
for eta in (0.01, 0.1, 0.25):
    bound = f(x) - eta * (1 - L * eta / 2) * (g @ g)
    print(f'eta = {eta:4.2f}:  bound {bound:7.4f},  actual {f(x - eta * g):7.4f}')

Very little, for the two stable steps: $6.0102$ under $6.012$, and $1.62$
under $1.8$. The third bound, $13.5$, is above the start $f = 7$, and the
step indeed raised $f$, to $12.375$. the figure
draws the two functions of $\eta$ behind these numbers. Along the step,
$\mathbf{g}^\top\mathrm{diag}(1, 10)\,\mathbf{g} = 1004$, so the objective is
the parabola $7 - 104\,\eta + 502\,\eta^2$, and the ceiling is
$7 - 104\,\eta + 520\,\eta^2$. The two are close because this gradient
points almost along the stiff axis, where the curvature is nearly $L$.

![The descent lemma on the opening example. Along $-\mathbf{g}$ from $(2,1)$ the objective is the lower parabola; the ceiling $f-\eta\|\mathbf{g}\|^2+\tfrac{L}{2}\eta^2\|\mathbf{g}\|^2$ is the upper one, tangent at $\eta=0$. Minimizing the ceiling ($\eta=1/L=0.1$) guarantees $f\le1.8$; the step achieves $1.62$. Past $2/L=0.2$ the ceiling is above the starting value and promises nothing; at $\eta=0.25$ the objective has risen to $12.4$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-descent-lemma-slice.svg)

The step $\eta = 1/L$ minimizes the ceiling, and $f$ lies below the ceiling:
that is all the guarantee says. that section
has the same argument in one variable.

### Stationary Points Without Convexity

The descent lemma alone, with no convexity, already says something about a
long run: the gradient cannot stay large.

**Proposition (gradient descent finds approximate stationary points).** *Let
$f$ be $L$-smooth and bounded below by $f_{\mathrm{low}}$, and run
the equation with $\eta = 1/L$. Then for every $K \ge 1$,*

$$
\min_{0 \le k < K} \|\nabla f(\mathbf{x}_k)\|^2 \;\le\; \frac{2L\,\big(f(\mathbf{x}_0) - f_{\mathrm{low}}\big)}{K}.
$$

**Proof.** At $\eta = 1/L$ the descent lemma reads
$\|\mathbf{g}_k\|^2 \le 2L\,(f(\mathbf{x}_k) - f(\mathbf{x}_{k+1}))$.
Sum over $k = 0, \ldots, K-1$; the right side telescopes to
$2L\,(f(\mathbf{x}_0) - f(\mathbf{x}_K)) \le 2L\,(f(\mathbf{x}_0) - f_{\mathrm{low}})$.
The smallest of $K$ numbers is at most their average. $\blacksquare$

Some iterate among the first $K$ has squared gradient norm $O(1/K)$, and
nothing beyond a lower bound and a smoothness constant was assumed. The
bound does need a global $L$. ReLU networks are not smooth, so for them it
describes an idealized model rather than a guarantee.

A small gradient does not mean a minimum. Take
$f(x, y) = \tfrac12 x^2 + \cos y$, which is bounded below by $-1$ and
$1$-smooth (its Hessian $\mathrm{diag}(1, -\cos y)$ has eigenvalues in
$[-1, 1]$). Gradient descent with $\eta = 1$ from $(1, 0)$ sets $x$ to $0$ in
one step and never moves $y$, because $\partial f/\partial y = -\sin y$
vanishes at $y = 0$. The run stops at $(0, 0)$, where $f = 1$. This point is
a saddle: $f$ rises along $x$ and falls along $y$. The minimum value $-1$ sits
at $(0, \pi)$. The bound holds, since the gradient at the saddle is zero, but
it says nothing about which stationary point is reached. Exercise
3 measures how long
runs from nearby starts stay near the saddle. For convex $f$ this cannot
happen: every stationary point is a global minimum
(that section).

### Backtracking Line Search

The step $\eta = 1/L$ presumes that we know $L$. We rarely do, and the
curvature that matters varies across the loss surface, so a single global
constant is both unknown and too conservative. A **line search** chooses the
step at run time. It tries a step and accepts it only if it achieves a fixed
fraction $c \in (0, 1)$ of the decrease that the first-order model predicts,
the **Armijo condition** (also called sufficient decrease)
[@Armijo.1966]

$$
f\big(\mathbf{x} - \eta\, \nabla f(\mathbf{x})\big) \;\le\; f(\mathbf{x}) - c\,\eta\, \|\nabla f(\mathbf{x})\|^2.
$$

The right side is the first-order prediction with its slope scaled down by
$c$. **Backtracking** starts from an optimistic $\eta_0$ and halves $\eta$
until the equation holds. By the descent lemma the condition
holds automatically once $1 - L\eta/2 \ge c$, that is, once
$\eta \le 2(1-c)/L$. So the loop stops after finitely many halvings and
accepts a step $\eta \ge \min\big(\eta_0,\, (1-c)/L\big)$, within a factor
two of that threshold, without ever knowing $L$.

The argument needs $L$ only on the region the method visits. Take the
quartic $f(x) = \tfrac14 x^4$, which has no global $L$, from $x_0 = 3$ with
$c = \tfrac12$. Backtracking never increases $f$, so every iterate stays in
$[-3, 3]$, where $f'' = 3x^2 \le 27$. A trial step with $\eta \le 1/27$ maps
$x$ to $x(1 - \eta x^2)$, which lies between $\tfrac23 x$ and $x$, so the
whole segment stays inside $[-3, 3]$ as well, and $L = 27$ there is all the
argument needs. The quartic's curvature is large far from the origin and
vanishes near it, so no single fixed step suits the whole trajectory. We
try both from this start: the fixed step $\eta = 0.3$, which would be
harmless near the origin, and backtracking from $\eta_0 = 1$.

In [ ]:
quartic = lambda v: 0.25 * (v ** 4).sum()
quartic_grad = lambda v: v ** 3

x = np.array([3.0])
for _ in range(6):                            # fixed step, too large at x0
    x = x - 0.3 * quartic_grad(x)
print(f'fixed eta = 0.3 from x0 = 3:  |x_6| = {abs(x[0]):.2e}  (diverged)')

x, etas = np.array([3.0]), []
for _ in range(25):                           # backtracking line search
    g = quartic_grad(x)
    eta = 1.0
    while quartic(x - eta * g) > quartic(x) - 0.5 * eta * (g * g).sum():
        eta *= 0.5                            # Armijo fails: halve the step
    x = x - eta * g
    etas.append(eta)
print('accepted steps:', np.array(etas[:8]), '...')
print(f'backtracking from x0 = 3:  f(x_25) = {quartic(x):.2e}  (monotone descent)')

The fixed step diverges: after six steps the iterate sits at
$|x_6| \approx 6.5 \times 10^{103}$. The update $x \mapsto x(1 - \eta x^2)$
shrinks $|x|$ if and only if $\eta x^2 < 2$, so from $x_0 = 3$ a fixed step
converges for $\eta < 2/9 \approx 0.22$ and diverges above it (exercise
4b). Note
that the threshold is not $2/f''(x_0) = 2/27$; that rule belongs to
quadratics. A step toward the origin meets the average of $f''$ between $0$
and $x$, which is $x^2$, not its largest value $3x^2$. The descent lemma has
to assume the largest and certifies only a third of the safe range.

Backtracking accepts $0.031$ on the first step. The halving before it,
$0.0625$, fails the Armijo test, which on this function holds if and only
if $\eta x^2 \le 0.456$, which at $x_0$ means $\eta$ up to about $0.05$. As
the iterate approaches the origin the accepted steps grow,
$0.0625, 0.125, 0.25, 0.5, 1.0$, each close to $0.3/x^2$: the line search
measures the curvature by trial. Training rarely backtracks, because a
minibatch loss is too noisy to compare two steps and each probe costs a
forward pass. Practice fixes the step in advance by a schedule
(that section).

## The Quadratic Model and the Condition Number

The opening promised to explain the difference between $69$ steps and
$6{,}908$. Both functions are quadratics, and on a quadratic gradient descent
can be solved exactly. The answer depends on a single number, the ratio of the
largest to the smallest curvature.

### Ill-Conditioned Valleys

the figure shows gradient descent on two
quadratics before any algebra. On a round bowl the path runs nearly straight
to the minimum. In a long valley the walls are steep and the floor is flat.
The step that the steep walls allow makes the iterate overshoot across the
valley and alternate from wall to wall. Along the flat floor the same step
barely moves, and the path zigzags. (The valley of the figure is
$\mathrm{diag}(1, 20)$ with $\eta = 0.097$, just below its ceiling $0.1$,
which gives the two factors in the caption.)

![Gradient descent on two quadratics $f(\mathbf{x})=\tfrac12\mathbf{x}^\top\mathbf{A}\mathbf{x}$. Left: a well-conditioned bowl; the path runs nearly straight to the minimum. Right: an ill-conditioned valley; a step near the stability ceiling makes the steep mode overshoot and alternate (factor $-0.94$) while the slow axis contracts by only $0.90$ per step.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-gd-bowl-vs-valley.svg)

Quadratics are worth solving because every smooth function looks like one
near a minimum. Near a minimizer $\mathbf{x}^\star$ with positive-definite
Hessian $\mathbf{A}$, the second-order Taylor expansion says that $f$ is
approximately
$f(\mathbf{x}^\star) + \tfrac12 (\mathbf{x}-\mathbf{x}^\star)^\top \mathbf{A}\, (\mathbf{x}-\mathbf{x}^\star)$.
Moving the minimizer to the origin and dropping the constant leaves the model
problem

$$
f(\mathbf{x}) = \tfrac12\, \mathbf{x}^\top \mathbf{A}\, \mathbf{x},
\qquad \mathbf{A} \textrm{ symmetric positive definite, with eigenvalues }
0 < \lambda_{\min} = \lambda_1 \le \cdots \le \lambda_d = \lambda_{\max}.
$$

Its gradient is $\mathbf{A}\mathbf{x}$, its Hessian is $\mathbf{A}$, and so
its smoothness constant is $L = \lambda_{\max}$. The ratio of the largest to
the smallest curvature,

$$
\kappa = \frac{\lambda_{\max}}{\lambda_{\min}},
$$

is the **condition number** of the problem. It is $10$ for
$\mathrm{diag}(1, 10)$ and $10^3$ for the chapter's running example. For a
symmetric positive definite
$\mathbf{A}$ it equals the singular-value ratio $\kappa(\mathbf{A})$ of
that section.

### Per-Mode Contraction and the Stability Ceiling

The zigzag in the valley comes from a ceiling on the step that the steep
direction imposes. To see where that ceiling lies, and what it costs the
flat direction, we solve gradient descent on a quadratic exactly.

Start with one variable, $f(u) = \tfrac{\lambda}{2} u^2$ with $\lambda > 0$.
The gradient step is $u_{k+1} = u_k - \eta\lambda u_k$, so

$$
u_{k+1} = (1 - \eta\lambda)\, u_k, \qquad u_k = (1 - \eta\lambda)^k\, u_0 .
$$

Every step multiplies the error by the same factor. The iterates converge if
and only if $|1 - \eta\lambda| < 1$, that is, $0 < \eta < 2/\lambda$. They
land on $0$ in one step if and only if $\eta = 1/\lambda$. For
$1/\lambda < \eta < 2/\lambda$ the factor is negative, and the iterate
alternates in sign while it shrinks. With $\lambda = 10$: $\eta = 0.1$ gives
the factor $0$, $\eta = 0.19$ gives $-0.9$, and $\eta = 0.21$ gives $-1.1$,
which diverges.

The general quadratic keeps this form. Its gradient is $\mathbf{A}\mathbf{x}$,
so the iteration the equation is

$$
\mathbf{x}_{k+1} = (\mathbf{I} - \eta \mathbf{A})\, \mathbf{x}_k .
$$

Every step applies the same linear map, so the whole run is the matrix power
$\mathbf{x}_k = (\mathbf{I} - \eta\mathbf{A})^k\mathbf{x}_0$. A symmetric
matrix has an orthonormal basis of eigenvectors
(that section). Write $\mathbf{x}_k$ in the eigenbasis
of $\mathbf{A}$ with coordinates $u_i^{(k)}$. The matrix
$\mathbf{I} - \eta\mathbf{A}$ has the same eigenvectors, with eigenvalues
$1 - \eta\lambda_i$, so the coordinates evolve independently:

$$
u_i^{(k+1)} = (1 - \eta \lambda_i)\, u_i^{(k)},
\qquad \textrm{hence} \qquad
u_i^{(k)} = (1 - \eta \lambda_i)^k\, u_i^{(0)}.
$$

Gradient descent on a quadratic is $d$ copies of the one-variable recursion,
one per eigenvalue, or **mode**. The factors $1 - \eta\lambda_i$ decide
convergence and speed:

* **Convergence** requires $|1 - \eta \lambda_i| < 1$ for every mode, that is,
  $0 < \eta < 2/\lambda_{\max} = 2/L$. This is the **stability ceiling**. On
  quadratics, the $2/L$ that the descent lemma offered as sufficient is also
  necessary. Above it, the stiffest mode's factor is below $-1$, so that
  coordinate alternates with growing amplitude even while the other modes
  converge (exercise
  4b builds
  such a half-converging, half-exploding run).
* **Speed** is set by the slowest mode. The error norm shrinks like
  $\rho(\eta)^k$, where $\rho(\eta) = \max_i |1 - \eta\lambda_i|$ is the
  **spectral radius** of $\mathbf{I} - \eta\mathbf{A}$, its largest eigenvalue
  magnitude.

On $\mathbf{A} = \mathrm{diag}(1, 10)$ the ceiling is $\eta = 0.2$. Before
reading the sweep below, guess which step leaves the smallest error after
$60$ steps from $(1, 1)$: the cautious $0.02$, the step $0.1 = 1/L$ that
the descent lemma favoured, or one closer to the ceiling.

In [ ]:
A = np.diag([1.0, 10.0])                      # eigenvalues 1 and 10, so L = 10
x0 = np.array([1.0, 1.0])

def run_gd(A, x0, eta, steps):
    xs = [x0]
    for _ in range(steps):
        xs.append(xs[-1] - eta * (A @ xs[-1]))
    return np.array(xs)

print(' eta    |1-eta*1|  |1-eta*10|  spectral radius   |x_60|')
for eta in [0.02, 0.10, 2 / 11, 0.19, 0.20, 0.21]:
    f1, f10 = abs(1 - eta * 1.0), abs(1 - eta * 10.0)
    print(f'{eta:5.3f}    {f1:6.3f}     {f10:6.3f}        {max(f1, f10):6.3f}     '
          f'{np.linalg.norm(run_gd(A, x0, eta, 60)[-1]):9.1e}')

The cautious $\eta = 0.02$ is stable but slow ($\rho = 0.98$; after $60$
steps the error is still $0.3$). At $\eta = 0.1$ the stiff mode is eliminated
in one step, yet the slow mode sets the rate, $\rho = 0.9$. The step
$\eta = 0.19$ also has $\rho = 0.9$, for the opposite reason: its stiff mode
alternates with factor $-0.9$. Between them, $\eta = 2/11 \approx 0.182$
balances the two extreme modes and wins by two orders of magnitude
($\rho = 0.818$, error $8.3 \times 10^{-6}$). At the ceiling
$\eta = 0.2$ the stiff mode's factor is $-1$: the iterate alternates between
$\pm 1$ on that axis, and $\|\mathbf{x}_{60}\| = 1.0$. At $\eta = 0.21$, sixty
steps increase the error to $3 \times 10^{2}$.

### The Optimal Step

Within the stable range, which fixed $\eta$ is fastest? The sweep suggests
balancing the two extreme modes. For fixed $\eta$, the function
$\lambda \mapsto |1 - \eta\lambda|$ is a V with its vertex at
$\lambda = 1/\eta$, so on $[\lambda_{\min}, \lambda_{\max}]$ its maximum sits
at an endpoint, and
$\rho(\eta) = \max\big(|1 - \eta\lambda_{\min}|,\, |1 - \eta\lambda_{\max}|\big)$.
Once $\eta$ exceeds $1/\lambda_{\max}$, raising it further lowers the value
at $\lambda_{\min}$ and raises the one at $\lambda_{\max}$.

**Proposition (optimal fixed step on a quadratic).** *For
$f(\mathbf{x}) = \tfrac12 \mathbf{x}^\top \mathbf{A} \mathbf{x}$ the convergence factor
$\rho(\eta) = \max_i |1 - \eta\lambda_i|$ is minimized by*

$$
\eta^\star = \frac{2}{\lambda_{\min} + \lambda_{\max}},
\qquad
\rho(\eta^\star) = \frac{\kappa - 1}{\kappa + 1},
\qquad \kappa = \frac{\lambda_{\max}}{\lambda_{\min}}.
$$

**Proof.** For every $\eta$,
$\rho(\eta) \ge \max(1 - \eta\lambda_{\min},\, \eta\lambda_{\max} - 1)$, and a
maximum of two numbers is at least any weighted average of them. Take the
weights $\lambda_{\max}/(\lambda_{\min}+\lambda_{\max})$ and
$\lambda_{\min}/(\lambda_{\min}+\lambda_{\max})$. The terms in $\eta$ cancel:

$$
\rho(\eta) \;\ge\; \frac{\lambda_{\max}(1 - \eta\lambda_{\min}) + \lambda_{\min}(\eta\lambda_{\max} - 1)}{\lambda_{\min} + \lambda_{\max}}
= \frac{\lambda_{\max} - \lambda_{\min}}{\lambda_{\max} + \lambda_{\min}} = \frac{\kappa - 1}{\kappa + 1}.
$$

Equality requires the two terms to be equal, $1 - \eta\lambda_{\min} = \eta\lambda_{\max} - 1$,
which is $\eta = \eta^\star$. At $\eta^\star$ both endpoint values are
$(\kappa - 1)/(\kappa + 1)$, and the V keeps every interior eigenvalue below
them, so the bound is attained. $\blacksquare$

the figure is this proof drawn on $\mathrm{diag}(1, 10)$:
the best V is the one whose endpoints are level. There
$\eta^\star = 2/11$ and $\rho = 9/11 = 0.818$, so six digits take
$\log 10^6/\log(11/9) \approx 69$ steps. On the chapter's running example
$\rho = 999/1001$, and six digits take $6{,}908$ steps.

![The per-mode contraction factors $|1-\eta\lambda|$ form a V with its vertex at $\lambda = 1/\eta$, and the convergence factor $\rho(\eta)$ is the larger of its values at the extreme eigenvalues $\lambda_{\min} = 1$, $\lambda_{\max} = 10$. The choice $\eta = 0.1$ eliminates the stiff mode but leaves the slow mode contracting at $0.9$; the optimal $\eta^\star = 2/(\lambda_{\min} + \lambda_{\max})$ equalizes the two endpoint factors at $(\kappa-1)/(\kappa+1) = 9/11$. Lowering either endpoint would raise the other.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-eta-tent.svg)

When $\kappa = 1$ (a round bowl) the rate is $0$ and one step solves the
problem. As $\kappa$ grows, $(\kappa-1)/(\kappa+1) \approx 1 - 2/\kappa$, so
$\log(1/\rho) \approx 2/\kappa$, and reducing the error by a factor $\epsilon$
costs about $\tfrac{\kappa}{2}\log\tfrac1\epsilon$ iterations: **the cost of
gradient descent is linear in the condition number**. Stability limits the
step by the stiff mode, $\eta \lesssim 2/\lambda_{\max}$, and with that step
the slow mode moves by only $\eta\,\lambda_{\min} \approx 2/\kappa$ per step.

At $\eta^\star$ the factor $\rho$ is more than an asymptotic rate. On
$\mathrm{diag}(1, 10)$ the two per-mode factors are $+9/11$ and $-9/11$ (the
row $\eta = 2/11$ of the sweep): the slow mode contracts monotonically while
the stiff mode alternates in sign. Both modes shrink by the same factor, so
$\|\mathbf{x}_k\|$ shrinks by $(\kappa-1)/(\kappa+1)$ at every single step,
not merely on average.

### The Edge of Stability

The stability ceiling suggests a recipe: measure the **sharpness**, the
largest Hessian eigenvalue $\lambda_{\max}(\nabla^2 f)$, and choose
$\eta < 2/\lambda_{\max}$. On a network this is not enough, because the
Hessian changes along the run. Under full-batch gradient descent the
sharpness typically rises until it reaches about $2/\eta$ and then stays
there, while the loss keeps falling over the run but rises on many single
steps [@Cohen.Kaur.Li.ea.2021]. that section runs
the experiment on a small $\tanh$ network and discusses it. Here we ask
only how much of it the quadratic model explains.

It explains half. In the eigenbasis of the Hessian the top mode's factor is
$1 - \eta\lambda_{\max}$, which is below $-1$ exactly when
$\lambda_{\max} > 2/\eta$. That coordinate then grows and alternates in
sign, which is how a loss that falls over the run can rise on single steps.
On a fixed quadratic the growth would end in divergence. Why it brings the
sharpness back down instead lies outside the model: the third-order term of
the Taylor expansion, which the quadratic model drops, turns the
oscillation into a drift toward lower sharpness
[@Damian.Nichani.Lee.2023]. Exercise
9 shows
that with momentum the ceiling moves to $(2 + 2\beta)/\eta$.

### From Quadratics to Convex Functions

What survives beyond quadratics is the linear dependence on $\kappa$. If
$f$ is $L$-smooth and $\mu$-strongly convex
the equation, it is sandwiched at every point
between quadratics of curvature $\mu$ and $L$, the ratio $\kappa = L/\mu$
takes the place of $\lambda_{\max}/\lambda_{\min}$, and gradient descent
with $\eta = 1/L$ satisfies

$$
f(\mathbf{x}_k)-f^\star \le \Big(1-\frac{1}{\kappa}\Big)^k\big(f(\mathbf{x}_0)-f^\star\big),
$$

which again means $O(\kappa\log\tfrac1\epsilon)$ steps. A function that is
convex and smooth but not strongly convex still reaches its optimal value,
at the slower rate $O(1/k)$, and without convexity only the stationarity
bound the equation remains. The proofs need
convexity and are in that section, which also
compares these bounds with what the quadratic achieves.

## Momentum and Acceleration

Gradient descent needs a number of steps proportional to $\kappa$ because one
step size must serve both the stiff and the flat modes. Momentum keeps the
gradient and changes the update so that the two modes can move at the same
rate.

### Heavy-Ball Momentum

The valley suggests the change. In the stiff direction successive gradients
point in alternating directions (the overshoot of
the figure), so averaging recent gradients would
cancel the oscillation. In the slow direction successive gradients are small
but agree, so accumulating them would build up speed. One mechanism does both:
give the iterate a velocity with memory. **Heavy-ball momentum**
[@Polyak.1964] replaces the gradient step by

$$
\mathbf{v}_{k+1} = \beta\, \mathbf{v}_k - \eta\, \nabla f(\mathbf{x}_k),
\qquad
\mathbf{x}_{k+1} = \mathbf{x}_k + \mathbf{v}_{k+1},
$$

with momentum parameter $\beta \in [0, 1)$. The velocity $\mathbf{v}_k$ is a
decaying sum of past gradients. Since $\mathbf{v}_k = \mathbf{x}_k - \mathbf{x}_{k-1}$,
eliminating it gives

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \eta\, \nabla f(\mathbf{x}_k) + \beta\,(\mathbf{x}_k - \mathbf{x}_{k-1}).
$$

Each step is a gradient step plus $\beta$ times the previous step; with
$\beta = 0$ it is gradient descent.

On the quadratic each mode again evolves on its own. With gradient $\lambda u_k$
in one mode, the eliminated form reads
$u_{k+1} = (1 + \beta - \eta\lambda)\, u_k - \beta\, u_{k-1}$, so the new value
depends on the last two. A second-order recurrence of this kind is the
discrete version of a damped oscillator, a mass on a spring with friction
$1 - \beta$, and $\beta$ sets the damping.

the figure shows the three regimes on
$\mathrm{diag}(1, 10)$, all with the gradient-descent step $\eta = 2/11$.
With too little momentum the path zigzags much like gradient descent. With
the right amount it settles after one overshoot. With too much, the iterate
loops around the minimum. The next subsection attaches a rate to each
regime and shows that the best rate of all needs a larger step as well.

![Heavy ball on the $\kappa=10$ valley $\tfrac12(x^2+10y^2)$, all three runs with the gradient-descent step $\eta=2/11$. Too little momentum zigzags like gradient descent (the stiff mode's coefficient $1+\beta-\eta\lambda_{\max}$ is negative, so a root is negative and real); the critically damped $\beta$ for this step is fastest; too much momentum loops around the minimum. The $\sqrt\kappa$ rate $0.52$ needs the larger step $\eta_{\mathrm{HB}}$ of the equation, at which $\beta=0.05$ would diverge.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-momentum-damping.svg)

### The $\sqrt{\kappa}$ Law

We now show that with the right pair $(\eta, \beta)$ every mode contracts at
the same rate $(\sqrt\kappa - 1)/(\sqrt\kappa + 1)$. For $\kappa = 10$ the pair
is $\beta^\star \approx 0.27$, $\eta_{\mathrm{HB}} \approx 0.23$, and the rate is
$0.52$ against gradient descent's $0.82$: six digits in $27$ steps instead of
$69$. Everything follows from one fact about a single mode.

**Proposition (heavy ball contracts every mode at $\sqrt\beta$).** *Let
$0 < \beta < 1$. On $f(u) = \tfrac{\lambda}{2}u^2$ the heavy-ball iterates
satisfy $u_{k+1} = (1 + \beta - \eta\lambda)u_k - \beta u_{k-1}$. If
$(1 - \sqrt\beta)^2 \le \eta\lambda \le (1 + \sqrt\beta)^2$, both roots of the
characteristic polynomial have modulus $\sqrt\beta$, and
$|u_k| \le C\,k\,\beta^{k/2}$ for $k \ge 1$ and a constant $C$ that depends on
$u_0$, $u_1$ and $\eta\lambda$. The factor $k$ is needed only at the two
endpoints, where the root is double.*

**Proof.** Substitute $u_k = z^k$ into the recurrence and divide by $z^{k-1}$:

$$
z^2 - (1 + \beta - \eta\lambda)\, z + \beta = 0.
$$

The product of the two roots is $\beta$. The roots are complex exactly when the
discriminant $(1 + \beta - \eta\lambda)^2 - 4\beta$ is negative, that is, when
$|1 + \beta - \eta\lambda| < 2\sqrt\beta$, which rearranges to
$(1 - \sqrt\beta)^2 < \eta\lambda < (1 + \sqrt\beta)^2$. Complex-conjugate roots
$z, \bar z$ with $z\bar z = \beta$ each have modulus $\sqrt\beta$, whatever
$\lambda$ is, and the general solution $u_k = a z^k + b \bar z^k$ is bounded by
$(|a| + |b|)\beta^{k/2}$. At the two endpoints the discriminant is zero, and
the double root is $(1 + \beta - \eta\lambda)/2 = \pm\sqrt\beta$. A double root
$z$ gives the general solution $u_k = (a + bk)z^k$, which is the factor $k$.
$\blacksquare$

**Corollary (the tuning).** Place the two extreme modes at the two ends of the
interval: $\eta\lambda_{\min} = (1 - \sqrt\beta)^2$ and
$\eta\lambda_{\max} = (1 + \sqrt\beta)^2$. Every other eigenvalue then lies
inside, so every mode has modulus $\sqrt\beta$. Taking square roots,
$\sqrt{\eta\lambda_{\max}} - \sqrt{\eta\lambda_{\min}} = 2\sqrt\beta$ and
$\sqrt{\eta\lambda_{\max}} + \sqrt{\eta\lambda_{\min}} = 2$. The second
equation gives $\eta$, and the ratio of the two gives $\sqrt\beta = (\sqrt\kappa - 1)/(\sqrt\kappa + 1)$.
With the tuning

$$
\eta_{\mathrm{HB}} = \left(\frac{2}{\sqrt{\lambda_{\max}} + \sqrt{\lambda_{\min}}}\right)^{\!2},
\qquad
\beta^\star = \left(\frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1}\right)^{\!2},
$$

heavy ball contracts every mode by the factor

$$
\rho_{\mathrm{HB}} = \sqrt{\beta^\star} = \frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1}
\;\approx\; 1 - \frac{2}{\sqrt{\kappa}}.
$$

This is gradient descent's rate with $\kappa$ replaced by $\sqrt\kappa$, so
the iteration count drops from $O(\kappa \log \tfrac1\epsilon)$ to
$O(\sqrt{\kappa} \log \tfrac1\epsilon)$ [@Polyak.1964]. At finite $k$
the double roots of the two extreme modes add a term. The error behaves like
$k\rho^k$, and with $\log(1/\rho) \approx 2/\sqrt\kappa$ the count for a target
$\epsilon$ solves $k \approx \tfrac{\sqrt\kappa}{2}\big(\log\tfrac1\epsilon + \log k\big)$,
the asymptotic law plus a term of order $\sqrt\kappa\log\kappa$.

the figure draws the proof for $\kappa = 10$: the
interval on which the modulus is flat at $\sqrt\beta$, and the roots
themselves, every pair on the circle of radius $\sqrt{\beta^\star} = 0.52$,
while gradient descent's factors spread along the real axis up to
$\pm 0.82$. Momentum lets the stiff and the flat mode share one complex modulus
instead of forcing the flat mode to crawl at $1 - 2/\kappa$.

The proposition also gives the rates of the three runs in
the figure, whose step puts the two modes at
$\eta\lambda = 0.18$ and $1.82$. For $\beta = 0.85$ both lie inside the
interval: the rate is $\sqrt{0.85} = 0.92$, and the complex roots make the
iterate loop. The smallest $\beta$ whose interval still holds the slow
mode, $\beta = (1 - \sqrt{2/11})^2 \approx 0.33$, is the fastest for this
step, with rate $0.57$. For $\beta = 0.05$ the interval $[0.60, 1.50]$ holds
neither mode. The slow one has the real root $0.81$, and the stiff one a
negative real root, which makes the iterate alternate, something a physical
oscillator never does.

![Heavy ball at the tuning of the equation on the $\kappa=10$ example. (a) For fixed $\beta$ the modulus of the larger root is flat at $\sqrt\beta$ wherever the roots are complex; the tuning places the two extreme modes at the ends of that interval. (b) In the complex plane every mode's root pair (labeled by its eigenvalue $\lambda$) lies on the circle of radius $\sqrt{\beta^\star}=0.52$, while gradient descent's factors spread over $[-0.82,0.82]$ on the real axis. The extreme modes are double roots, which is the factor $k$ in the error.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-opt-hb-eigenvalues.svg)

The law is exact for quadratics only. @Lessard.Recht.Packard.2016
constructed a one-dimensional, smooth, strongly convex function on which heavy
ball with this tuning does not converge from some starting points: the
iterates fall into a stable cycle (exercise
7). Momentum
in deep learning [@Sutskever.Martens.Dahl.ea.2013] rests on the local
quadratic model and on experience, not on a global theorem.

### Nesterov's Look-Ahead

A variant that does carry a global guarantee evaluates the gradient at a
look-ahead point displaced by the momentum. **Nesterov's accelerated
gradient** [@Nesterov.1983] is

$$
\mathbf{v}_{k+1} = \beta\, \mathbf{v}_k - \eta\, \nabla f(\mathbf{x}_k + \beta\, \mathbf{v}_k),
\qquad
\mathbf{x}_{k+1} = \mathbf{x}_k + \mathbf{v}_{k+1}.
$$

The only change from heavy ball is where the gradient is taken. When the
momentum is about to overshoot, the gradient at the look-ahead point already
points back and brakes the step.

**Proposition (Nesterov on quadratics).** *Let $\kappa = L/\mu$. With
$\eta = 1/L$ and $\beta = (\sqrt\kappa - 1)/(\sqrt\kappa + 1)$, every mode of a
quadratic with eigenvalues in $[\mu, L]$ has roots of modulus at most
$1 - 1/\sqrt\kappa$, with a double root at the slow mode $\lambda = \mu$, so
the error is at most $C\,k\,(1 - 1/\sqrt\kappa)^k$.*

**Proof.** In one mode with eigenvalue $\lambda$, write $a = 1 - \eta\lambda$,
which lies in $[0, 1 - 1/\kappa]$. The velocity update becomes
$v_{k+1} = a\beta v_k - \eta\lambda u_k$, and $u_{k+1} = a u_k + a\beta v_k$;
substituting $v_k = u_k - u_{k-1}$ gives
$u_{k+1} = (1 + \beta)a\,u_k - \beta a\,u_{k-1}$. The characteristic polynomial
$z^2 - (1 + \beta)a\,z + \beta a$ has discriminant $a\,[(1 + \beta)^2 a - 4\beta]$.
With this $\beta$, $(1 + \beta)^2 = 4\kappa/(\sqrt\kappa + 1)^2$. At the slow
mode, $a = (\kappa - 1)/\kappa$, the bracket is
$4(\kappa - 1)/(\sqrt\kappa + 1)^2 - 4\beta = 4\beta - 4\beta = 0$: a double
root at $(1 + \beta)a/2 = (\sqrt\kappa - 1)/\sqrt\kappa = 1 - 1/\sqrt\kappa$.
Every stiffer mode has a smaller $a$, a negative bracket, and complex roots
(or roots $0$ when $a = 0$) of modulus $\sqrt{\beta a} < \sqrt{\beta(1 - 1/\kappa)} = 1 - 1/\sqrt\kappa$.
$\blacksquare$

The rate is $0.684$ for $\kappa = 10$, $0.900$ for $\kappa = 100$, and $0.968$
for $\kappa = 1000$, slower than heavy ball's $0.519$, $0.818$, and $0.939$,
but with the same $\sqrt\kappa$ dependence. Unlike heavy ball, Nesterov's
method keeps its guarantees beyond quadratics, which we state without proof
[@Nesterov.2018]. For $\mu$-strongly convex, $L$-smooth $f$, the method
with the same constant $\beta$ satisfies
$f(\mathbf{x}_k) - f^\star \le C\,(1 - 1/\sqrt\kappa)^k$ on every function of
the class. Mind the units: this bounds the gap $f - f^\star$, which scales
as the square of the distance, so it has half the exponent that the
proposition gives on a quadratic. For smooth convex $f$ a
schedule $\beta_k \to 1$ gives $f(\mathbf{x}_k) - f^\star = O(1/k^2)$,
against gradient descent's $O(1/k)$. No method that forms its iterates from
past gradients can beat $O(1/k^2)$ or the $\sqrt\kappa$ dependence in the
worst case, as long as the dimension is large compared with the number of
steps [@Nemirovski.Yudin.1983]. Exercise
8 builds
the worst function.

It remains to collect the counts that the opening promised. On
$\mathrm{diag}(1, \kappa)$ from $(1, 1)$, six digits should cost gradient
descent $\tfrac{\kappa}{2}\log 10^6 \approx 6.9\,\kappa$ steps and heavy ball
somewhat more than $6.9\sqrt\kappa$.

In [ ]:
def iterations(lam, eta, beta=0.0, lookahead=False, tol=1e-6):
    """Steps until |x_k| <= tol |x_0| on f(x) = sum_i lam_i x_i^2 / 2."""
    x, v = np.array([1.0, 1.0]), np.zeros(2)
    n0 = np.linalg.norm(x)
    for k in range(1, 10**6):
        g = lam * (x + beta * v) if lookahead else lam * x
        v = beta * v - eta * g
        x = x + v
        if np.linalg.norm(x) <= tol * n0:
            return k

print('kappa      GD   heavy ball   Nesterov     GD/kappa   HB/sqrt(kappa)')
for kappa in [10.0, 100.0, 1000.0]:
    lam, sk = np.array([1.0, kappa]), np.sqrt(kappa)
    gd = iterations(lam, 2 / (1 + kappa))
    hb = iterations(lam, (2 / (1 + sk))**2, ((sk - 1) / (sk + 1))**2)
    na = iterations(lam, 1 / kappa, (sk - 1) / (sk + 1), lookahead=True)
    print(f'{kappa:5.0f} {gd:7d} {hb:9d} {na:10d}      '
          f'{gd / kappa:7.1f}     {hb / sk:7.1f}')

Gradient descent's count divided by $\kappa$ sits at $6.9$, so it needs
$6{,}908$ steps at $\kappa = 1000$. Heavy ball needs $315$ there. Its count
divided by $\sqrt\kappa$ rises, $8.5$, $9.3$, $10.0$, because each extreme
mode is a double root. Its $2\times 2$ iteration matrix is then a Jordan block, a repeated
eigenvalue with a single eigenvector (that section), whose
$k$-th power carries the factor $k$. The formula
$k \approx \tfrac{\sqrt\kappa}{2}(\log 10^6 + \log k)$ predicts $8.6$, $9.2$,
$9.8$, and at $\kappa = 10^4$ heavy ball needs $1{,}056$ steps, ratio $10.6$
(exercise 6).
Nesterov's $508$ at $\kappa = 1000$ sits between the two, as its rate $0.968$
against heavy ball's $0.939$ predicts. All three methods use one gradient per
step, so momentum's speedup costs no extra gradients. The square root is
also as far as gradients will go: by the lower bound, no method built from
past gradients does better in high dimension. To remove $\kappa$ altogether
a step has to know the curvature (that section).

## Summary

A gradient step can be trusted up to
$\eta = 2/L$. On a quadratic the best fixed step contracts the error by
$(\kappa - 1)/(\kappa + 1)$ per step, so the number of steps grows linearly
with the condition number $\kappa$: stability is set by the stiffest mode
and progress by the flattest. Momentum makes every mode contract at the
same rate $\sqrt\beta$, and its tuned rate
$(\sqrt\kappa - 1)/(\sqrt\kappa + 1)$ cuts the count to order $\sqrt\kappa$.
On a network the sharpness moves to the ceiling $2/\eta$ of the chosen
step, which the quadratic model explains only in part. Three questions
remain open: what a step can do when it knows the curvature
(that section), when a stationary point is the global
minimum (that section), and what noisy gradients cost
(that section).

## Exercises

1. **Preconditioned descent directions.** The proposition of
   that section measures the length of a
   direction by the Euclidean norm. Other norms give other steepest
   directions.
    1. Show that $\mathbf{d} = -\mathbf{P}^{-1}\nabla f(\mathbf{x})$ is a descent
       direction for every positive definite matrix $\mathbf{P}$.
    1. Measure length by
       $\|\mathbf{d}\|_{\mathbf{P}} = \sqrt{\mathbf{d}^\top \mathbf{P}\, \mathbf{d}}$. Show that
       the minimizer of $\nabla f(\mathbf{x})^\top \mathbf{d}$ over
       $\|\mathbf{d}\|_{\mathbf{P}} = 1$ is proportional to
       $-\mathbf{P}^{-1}\nabla f(\mathbf{x})$, and find the $\mathbf{P}$ for which it is
       Newton's direction the equation. The substitution
       $\mathbf{u} = \mathbf{P}^{1/2}\mathbf{d}$ reduces the problem to the Euclidean
       case.

1. [code] **Smoothness constants.** The smoothness constant $L$ of
   the equation bounds the curvature of $f$, and for a
   logistic loss it can be computed from the data.
    1. Let $f$ be twice continuously differentiable and $L$-smooth. Show that
       $\|\nabla^2 f(\mathbf{x})\,\mathbf{v}\| \le L\,\|\mathbf{v}\|$ for
       all $\mathbf{x}$ and $\mathbf{v}$, and conclude that every Hessian
       eigenvalue lies in $[-L, L]$. This is the converse direction of the
       equivalence stated after the equation.
    1. Let
       $f(\mathbf{w}) = \frac1n \sum_{i=1}^n \log\big(1 + \exp(-y_i\, \mathbf{x}_i^\top \mathbf{w})\big) + \frac{\nu}{2}\|\mathbf{w}\|^2$
       with labels $y_i = \pm 1$ and $\nu = 0.1$, the loss `loss` of the data
       cell of that section, where $\mathbf{x}_i^\top$ is the $i$-th
       row of $\mathbf{X}$. Show that
       $L = \lambda_{\max}(\mathbf{X}^\top \mathbf{X})/(4n) + \nu$ is a valid
       smoothness constant and that the Hessian attains this bound at
       $\mathbf{w} = \mathbf{0}$.
    1. Compute this $L$ for the data of that cell. It comes out near $0.4$
       for every draw; explain why, using
       $\lambda_{\max}(\mathbf{X}^\top\mathbf{X})/n \approx (1 + \sqrt{d/n})^2$
       for standard Gaussian entries. Count the steps that $\eta = 0.5$ (the
       reference's step) and $\eta = 1/L$ each need, from $\mathbf{w} = \mathbf{0}$,
       to reach $\|\nabla f(\mathbf{w})\| \le 10^{-10}$. Which is faster, and
       does $\eta = 0.5$ stay below $2/L$?

1. [code] **A saddle that attracts.** The stationarity guarantee
   the equation does not distinguish minima from
   saddle points. Consider $f(x, y) = \tfrac12 x^2 + \cos y$, which the text
   ran from $(1, 0)$ into the saddle $(0, 0)$.
    1. Show that $f$ is bounded below and $L$-smooth with $L = 1$. Find all
       stationary points and classify each one as a minimum or a saddle
       point.
    1. Run gradient descent the equation with $\eta = 1/L$
       from $(1, \delta)$ for
       $\delta \in \{10^{-2}, 10^{-4}, 10^{-8}, 10^{-12}\}$ and count the
       steps until $f < -0.99$. Linearize the update
       $y_{k+1} = y_k + \sin y_k \approx 2y_k$ near $y = 0$ to predict how
       this count grows as $\delta$ shrinks, and compare the prediction with
       your counts.
    1. What do these runs suggest about whether gradient descent from a
       random initialization converges to a saddle point? What do they
       suggest about the time it can spend near one?

1. [code] **Step size and the ceiling.** Let
   $f(\mathbf{x}) = \tfrac12\mathbf{x}^\top \mathbf{A}\,\mathbf{x}$ with $\mathbf{A}$
   symmetric positive definite and condition number $\kappa$. Suppose the
   step $\eta = 1/\hat{L}$ is computed from an estimate $\hat{L}$ of the
   smoothness constant $L = \lambda_{\max}$, and write
   $\alpha = L/\hat{L}$, so that $\eta = \alpha/\lambda_{\max}$.
    1. Use the equation to express the convergence factor
       $\rho(\eta) = \max_i |1 - \eta\lambda_i|$ through $\alpha$ and
       $\kappa$. Show that for $0 < \alpha \le 1$, reducing the error by a
       factor $\epsilon$ takes about
       $(\kappa/\alpha)\log(1/\epsilon)$ steps. What does overestimating
       $L$ by a factor of two cost? What does underestimating it by a factor
       of two cost?
    1. On the quartic $f(x) = \tfrac14 x^4$ from $x_0 = 3$, find the largest
       fixed step for which gradient descent converges. Show that
       $|x_{k+1}| < |x_k|$ if and only if $0 < \eta x_k^2 < 2$, conclude that
       every $\eta < 2/9$ converges, and show that at $\eta = 2/9$ the iterate
       flips between $3$ and $-3$ forever. Confirm both by a run. Backtracking
       accepted $0.031$ on the first step and $1.0$ later; explain why a fixed
       step must respect the curvature at the starting point for the whole
       run.

1. **Steepest descent in other norms.** ● Problem
   1b measures
   length by a norm $\|\cdot\|_{\mathbf{A}}$ that comes from an inner product. Norms
   that do not come from an inner product give steepest directions of a
   different form.
    1. Find the minimizers of $\nabla f(\mathbf{x})^\top \mathbf{d}$ over
       $\|\mathbf{d}\|_\infty = 1$ and over $\|\mathbf{d}\|_1 = 1$, together
       with the minimal slopes, and state when each minimizer is unique.
    1. Let the parameter be a matrix whose gradient $\mathbf{G}$ has the
       reduced SVD $\mathbf{G} = \mathbf{U}\, \mathrm{diag}(\sigma_i)\, \mathbf{V}^\top$,
       let the slope of a direction $\mathbf{D}$ be
       $\operatorname{tr}(\mathbf{G}^\top \mathbf{D})$, and measure length by the
       spectral norm $\|\mathbf{D}\|_2$. Show that the minimal slope over
       $\|\mathbf{D}\|_2 \le 1$ is $-\sum_i \sigma_i$, attained at
       $\mathbf{D} = -\mathbf{U} \mathbf{V}^\top$, the polar factor that Muon
       applies to its momentum matrix
       (that section). Writing
       $\operatorname{tr}(\mathbf{G}^\top \mathbf{D}) = \sum_i \sigma_i\, \mathbf{u}_i^\top \mathbf{D}\, \mathbf{v}_i$
       is a useful first step. This is the steepest-descent view of
       @Bernstein.Newhouse.2024 .

1. [code] **Heavy ball: the double root and the factor $k$.** ● Apply heavy
   ball the equation to $f(u) = \tfrac{\lambda}{2} u^2$.
    1. Write one step as a linear map of the state $(u_k, u_{k-1})$ and show
       that the characteristic polynomial of its $2 \times 2$ matrix is
       the equation of the text.
    1. At the tuning the equation the two extreme modes have
       the double eigenvalues $\sqrt{\beta^\star}$ and $-\sqrt{\beta^\star}$.
       Show that the matrix then has a single eigenvector, and that the
       $k$-th power of the Jordan block
       $\left(\begin{smallmatrix}\rho&1\\0&\rho\end{smallmatrix}\right)$ has
       the entry $k\rho^{k-1}$ above the diagonal.
    1. With $\|\mathbf{e}_k\| \approx k\rho^k$ and
       $\log(1/\rho) \approx 2/\sqrt\kappa$, solve
       $k = \tfrac{\sqrt\kappa}{2}\big(\log\tfrac1\epsilon + \log k\big)$
       numerically for $\epsilon = 10^{-6}$ and
       $\kappa \in \{10, 100, 1000\}$, and compare $k/\sqrt\kappa$ with the
       momentum cell's column $8.5$, $9.3$, $10.0$.
    1. Run the momentum cell's heavy ball at $\kappa = 10^4$ and compare its
       count with your prediction. Explain why the ratio $k/\sqrt\kappa$
       never settles as $\kappa$ grows.

1. [code] **Heavy ball beyond quadratics.** ●
   @Lessard.Recht.Packard.2016 define a one-dimensional function
   through its gradient,

   $$
   \nabla f(x) = \begin{cases} 25x, & x < 1, \\ x + 24, & 1 \le x < 2, \\ 25x - 24, & x \ge 2. \end{cases}
   $$

    1. Show that $f$ is $\mu$-strongly convex with $\mu = 1$ and $L$-smooth
       with $L = 25$.
    1. Run heavy ball with the tuning the equation for $300$
       steps from $x_0 = 3.3$ with zero initial velocity, and run Nesterov's
       method the equation with $\eta = 1/L$ and
       $\beta = (\sqrt{\kappa} - 1)/(\sqrt{\kappa} + 1)$ from the same
       start, where $\kappa = L/\mu$. Plot $x_k$ against $k$ for both
       methods.
    1. The heavy-ball iterates approach a cycle of period three that visits
       the piece $x < 1$ twice and the piece $x \ge 2$ once. Write the
       heavy-ball update on these pieces as a linear system for the three
       points of the cycle, solve it, and compare the solution with the
       simulation.
    1. Repeat the heavy-ball run from a grid of starting points on
       $[-6, 6]$ and report the fraction that end on the cycle. Every piece
       of $f$ is a quadratic with curvature in $[\mu, L]$. Why does the
       per-mode analysis behind the equation nevertheless not
       apply to $f$?

1. [code] **A lower bound for first-order methods.** ● This problem derives
   the $O(1/k^2)$ part of the bound of @Nemirovski.Yudin.1983 quoted
   in that section, the part for smooth convex
   functions, following the construction of @Nesterov.2018 . A
   first-order method here is any method whose iterates satisfy
   $\mathbf{x}_k \in \mathbf{x}_0 + \mathrm{span}\{\nabla f(\mathbf{x}_0), \ldots, \nabla f(\mathbf{x}_{k-1})\}$;
   take $\mathbf{x}_0 = \mathbf{0}$. For $p \le d$, let
   $\mathbf{A}_p \in \mathbb{R}^{d \times d}$ be zero except for its leading
   $p \times p$ block, which is tridiagonal with $2$ on the diagonal and $-1$
   on both off-diagonals, and let

   $$
   f_p(\mathbf{x}) = \frac{L}{4} \left( \tfrac12\, \mathbf{x}^\top \mathbf{A}_p\, \mathbf{x} - x_1 \right).
   $$

    1. Show that $f_p$ is convex and $L$-smooth.
    1. Show that if $\mathbf{x}$ is supported on the first $j$ coordinates,
       then $\nabla f_p(\mathbf{x})$ is supported on the first $j + 1$.
       Conclude that the $k$-th iterate of every first-order method is
       supported on the first $k$ coordinates.
    1. Show that the minimizer of $f_p$ has entries
       $x_i^\star = 1 - i/(p+1)$ for $i \le p$ and zero beyond, that the
       minimum is $-\frac{L}{8}\big(1 - \frac{1}{p+1}\big)$, and that
       $\|\mathbf{x}^\star\|^2 \le (p+1)/3$.
    1. Let $2k + 1 \le d$. The functions $f_{2k+1}$ and $f_k$ agree on
       vectors supported on the first $k$ coordinates. Use this fact to show
       that every first-order method applied to $f_{2k+1}$ satisfies, with
       $\mathbf{x}^\star$ the minimizer of $f_{2k+1}$,

       $$
       f_{2k+1}(\mathbf{x}_k) - \min f_{2k+1} \;\ge\; \frac{3L\, \|\mathbf{x}_0 - \mathbf{x}^\star\|^2}{32\,(k+1)^2}.
       $$

       The construction needs the dimension $d \ge 2k + 1$, so the bound
       governs about the first $d/2$ iterations. In a fixed dimension, long
       runs can do better asymptotically, but for networks with $d$ in the
       millions the restriction is inactive.
    1. For $k \in \{5, 10, 25, 50, 100, 200, 400\}$, run $k$ steps of
       gradient descent with $\eta = 1/L$ and of Nesterov's update
       the equation with $\eta = 1/L$ and
       $\beta_j = (j-1)/(j+2)$ at step $j = 1, 2, \ldots$ on $f_{2k+1}$ with
       $d = 2k + 1$, and divide each gap by the bound. Which ratio stays
       bounded as $k$ grows?

1. [code] **Edge of stability with momentum.** ●●
   @Cohen.Kaur.Li.ea.2021 report that with momentum, the sharpness
   settles at a threshold that differs from $2/\eta$.
    1. Use the iteration matrix of problem
       6a to
       show that heavy ball the equation on
       $f(x) = \tfrac{\lambda}{2} x^2$ with $0 \le \beta < 1$ converges if
       and only if $0 < \eta\lambda < 2 + 2\beta$. Derive the corresponding
       interval for Nesterov's update the equation.
    1. Modify the edge-of-stability experiment of
       that section (cell
       `#optimization-intro-the-edge-of-stability-1`) to train with heavy ball and with Nesterov's update for
       $\beta \in \{0.25, 0.5\}$, choosing for each run a step size whose
       predicted threshold lies between $5$ and $12$. Track the sharpness
       against $2/\eta$ and against the threshold from (a).
    1. At which threshold does the sharpness settle? What does the answer
       imply for choosing $\eta$ from a sharpness measured at
       initialization when training with momentum?

[Discussions](https://d2l.discourse.group/t/36828)